# Qwen2.5-1.5B zero-shot baseline: disconnect-safe re-run
**Runtime → Change runtime type → T4 GPU.** Keep this browser tab open while it runs.

Everything is saved to your Google Drive (folder `qwen_run`). **If Colab disconnects:** reconnect, then run the cells again from the top. The bundle is reloaded from Drive (no second upload), and finished seeds are skipped automatically.

Order: GHPR seeds 0, 1, 2 (the missing result), then PI-Link seed 0 as a check against the paper (MRR about 0.48). Each seed takes roughly 10–20 minutes.

In [ ]:
from google.colab import drive, files
import os, shutil
drive.mount('/content/drive')
D = '/content/drive/MyDrive/qwen_run'; os.makedirs(D, exist_ok=True)
if not os.path.exists(f'{D}/qwen_bundle.zip'):
    up = files.upload()  # choose qwen_bundle.zip (first time only)
    shutil.copy('qwen_bundle.zip', f'{D}/qwen_bundle.zip')
!unzip -oq {D}/qwen_bundle.zip -d /content/run && ls /content/run

In [ ]:
import torch; print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else 'NONE - switch the runtime to T4 GPU')

## GHPR, one seed per cell
Each cell ends with a line like `ghpr 0 AUC … MRR … (saved)`.

In [ ]:
%cd /content/run
!python -u qwen_rerun.py --datasets ghpr --seeds 0 --out {D}/results 2>&1 | grep -v -i warning

In [ ]:
%cd /content/run
!python -u qwen_rerun.py --datasets ghpr --seeds 1 --out {D}/results 2>&1 | grep -v -i warning

In [ ]:
%cd /content/run
!python -u qwen_rerun.py --datasets ghpr --seeds 2 --out {D}/results 2>&1 | grep -v -i warning

## PI-Link check (one seed)
Should be close to the paper's PI-Link value (seed 0 of the original run). Optional seeds 1 and 2 below if you have time.

In [ ]:
%cd /content/run
!python -u qwen_rerun.py --datasets pilink --seeds 0 --out {D}/results 2>&1 | grep -v -i warning

In [ ]:
# optional
%cd /content/run
!python -u qwen_rerun.py --datasets pilink --seeds 1 2 --out {D}/results 2>&1 | grep -v -i warning

## Download
Attach `qwen_results.zip` in the chat. The same files stay in Drive → `qwen_run/results`.

In [ ]:
!cd {D} && zip -qr /content/qwen_results.zip results
files.download('/content/qwen_results.zip')